In [1]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

dirs = Path().resolve().parent

if dir not in sys.path:
    sys.path.append(str(dirs))

Failed to read module file 'C:\Users\User\anaconda3\Lib\re\_casefix.py' for module 're._casefix': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\CodingHenry\research_MBKM\venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\CodingHenry\research_MBKM\venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\anaconda3\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1140, in _find_and_load_unlocked
ModuleNotFoundError: No mod

In [2]:
from utils.garch import simulate_garch
import joblib
import yfinance as yf
import math
from utils.utils import log_transform, split_into_windows
import numpy as np
from arch import arch_model

In [3]:
ticker = "^GSPC"
start_interval = "2010-01-01"
end_interval = "2026-01-01"
interval = "1d"

raw_snp500 = yf.Ticker(ticker).history(
    start=start_interval,
    end=end_interval,
    interval=interval
)["Close"].to_numpy()

n = len(raw_snp500)
split = math.ceil(n * 0.2)

train_end = n - 2 * split

train_raw_snp500 = raw_snp500[:train_end]

train_snp500 = log_transform(train_raw_snp500)

train_snp500 = train_snp500[~np.isnan(train_snp500)]

In [4]:
model = arch_model(
    train_snp500 * 100,
    mean="zero",
    vol="GARCH",
    p=1,
    q=1,
    dist="t"
)

res = model.fit(disp="off")

In [5]:
config = {
  "mean": "zero",
  "p": 1,
  "q": 1,
  "burn": 500
}

In [6]:
sim_data_128 = {
    "window_size": 128,
    "config": config.copy(),
    "sim_data": simulate_garch(res, 36, 128)
}

sim_data_256 = {
    "window_size": 256,
    "config": config.copy(),
    "sim_data": simulate_garch(res, 18, 256)
}

sim_data_512 = {
    "window_size": 512,
    "config": config.copy(),
    "sim_data": simulate_garch(res, 8, 512)
}

In [7]:
sim_data_128["sim_data"].shape, sim_data_256["sim_data"].shape, sim_data_512["sim_data"].shape,

((36, 128), (18, 256), (8, 512))

In [8]:
SYN_PATH_128 = dirs / "data" / "sim_data_128.joblib"
SYN_PATH_256 = dirs / "data" / "sim_data_256.joblib"
SYN_PATH_512 = dirs / "data" / "sim_data_512.joblib"
joblib.dump(sim_data_128, SYN_PATH_128)
joblib.dump(sim_data_256, SYN_PATH_256)
joblib.dump(sim_data_512, SYN_PATH_512)

['D:\\CodingHenry\\research_MBKM\\data\\sim_data_512.joblib']